# 1. Compare real population products

Run **Run → Run All Cells** in JupyterLite/Pyodide. The first kernel start retrieves the pinned Python runtime. Population calculations use bundled local source pixels and Python's standard library. New real-data notebook names and a versioned helper alias preserve returning users' older synthetic notebooks in browser storage.

We compare native Greater Lagos excerpts of WorldPop Global2 2025, GRID3 Nigeria 2025, LandScan Global 2020 and GHS-POP 2025. Prepared extent: `[3.2, 6.35, 3.6, 6.7]`; edge pixels extend to native boundaries. This is a regional excerpt, not global streaming. Density is only a map display; stored values are people per native cell.


In [ ]:
import json
from pathlib import Path
from population_tools_real import load_population, compare, validate_layer
data = load_population()
oracle = json.loads(Path('independent_oracle.json').read_text(encoding='utf-8'))
for layer in data['layers']:
    validate_layer(layer)
for layer in data['layers']:
    m = layer['metadata']
    print(m['title'], m['version'], m['reference_date'], m['population_concept'], m['license'])


## Reproduce the viewer rectangle

The shared WGS84 rectangle is `[3.30, 6.45, 3.45, 6.60]`. Independent NumPy/rasterio calculations from source pixels give approximately 3,058,259.42 WorldPop, 2,375,656.06 GRID3, 7,422,967.00 LandScan and 5,013,773.99 GHS-POP people, with different numeric-cell coverage. The code checks full precision against the separate source oracle.


In [ ]:
results = compare(data['layers'], data['default_aoi'])
for result in results:
    expected = oracle['sources'][result['metadata']['id']]
    assert abs(result['population'] - expected['aoi_population']) < .0001
    assert abs(result['coverage'] - min(1, expected['aoi_coverage'])) < 1e-10
    assert result['intersected_cells'] == expected['aoi_intersected_cells']
    print(result['metadata']['id'], round(result['population'], 2), f"coverage={result['coverage']:.2%}")
print('REAL_POPULATION_BROWSER_OK')


WorldPop refers to January 1, 2025; GRID3 is scaled to July 2025. LandScan is 2020 ambient (24-hour average) population; the others are residential estimates/projection. GHS-POP 2025 is a projection using GPWv4.11 demographic inputs. GRID3 and WorldPop share producers/inputs. Agreement is not independent validation, and disagreement does not determine accuracy.

Change the rectangle below; areas outside prepared grids reduce coverage instead of inventing values.


In [ ]:
my_bounds = [3.32, 6.48, 3.40, 6.56]
[(r['metadata']['title'], r['population'], r['coverage']) for r in compare(data['layers'], my_bounds)]


## Sources and software

The complete citation, release, native geometry, source asset, NoData semantics, license, retrieval date (2026-11-06) and processing receipt travel in every layer's metadata. These are native-pixel Greater Lagos excerpts, spatially subset and losslessly serialized without resampling; no producer endorsement. All four products are CC BY 4.0. WorldPop: Bondarenko et al. (2025), https://doi.org/10.5258/SOTON/WP00839 (Global2 R2025A v1 alpha); GRID3: Nnanatu et al. (2025), https://doi.org/10.5258/SOTON/WP00782; LandScan: Rose et al. (2021), https://doi.org/10.48690/1523378; GHS-POP: Schiavina et al. (2023), https://doi.org/10.2905/2FF68A52-5B5B-4A22-8F40-C41DA8332CFE and Pesaresi et al. (2024), https://doi.org/10.1080/17538947.2024.2390454. Credit WorldPop / University of Southampton; GRID3 Inc. and CIESIN / Columbia Climate School; Oak Ridge National Laboratory; European Commission/JRC, (c) European Union, 1995-2026. Gates Foundation WorldPop INV-045237 and GRID3 INV-044979. Preserve all producer notices when sharing.

LandScan pixels came from a pinned Hugging Face mirror, not an ORNL-hosted TIFF. Independent ORNL statistics matched one cell and the complete prepared window; they do not prove full mirror bit identity. GHS-POP uses the official JRC source, not a quantized third-party mirror. GPW is not bundled; current Harvard Dataverse access uses a guestbook. GRID3 and WorldPop file servers currently lack browser CORS/range support; download yourself and use the viewer's bounded local TIFF import for other areas.

Inspired by NASA Earthdata, SEDAC/CIESIN and POPGRID Viewer (https://www.earthdata.nasa.gov/news/feature-articles/data-tool-focus-popgrid-viewer). Fitness for use: Leyk et al. (2019), https://doi.org/10.5194/essd-11-1385-2019. Execution: Project Jupyter / JupyterLite, Pyodide, Jupyter Book / MyST. The viewer uses Leaflet 1.9.4 and GeoTIFF.js 3.0.5 with their notices. RustGIS, GeoLibre, JupyterGIS and FieldWork integration remain future work.
